# D2-01 ecoinvent import and Brightway recap

This notebook bridges the BAFU-based Brightway work from Day 1 with the `ecoinvent 3.12 cutoff` workflows needed later for `premise`, `Activity Browser`, and optional `trails`.


## Learning goals

- Switch back to the shared Paris Brightway project.
- Confirm that the Day 1 BAFU import is still available.
- Import `ecoinvent 3.12 cutoff` with the `bw2io` helper interface.
- Inspect the imported database and run one quick comparison LCA.
- Prepare the project state needed for Activity Browser Part I.


## Background references

- Revisit `D1-04` if you need a refresher on project setup, imports, and method inspection.
- The `ecoinvent` import uses the helper interface already previewed at the end of `D1-04`.


## 1) Switch to the shared project and inspect the current state

We keep using the same Brightway project across the whole course so that later notebooks can build on the Day 1 work.


In [1]:
import bw2data as bd
import pandas as pd

project_name = 'paris-lca-course-2026'
bd.projects.set_current(project_name)

databases = sorted(bd.databases)
pd.DataFrame({'database': databases})


/opt/homebrew/Caskroom/miniforge/base/envs/lca-course/lib/python3.11/site-packages/scikits/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__('pkg_resources').declare_namespace(__name__)


,database
0,bafu
1,carbon fiber
2,ecoinvent-3.10-biosphere


## 2) Import `ecoinvent 3.12 cutoff`

The cell below uses `bw2io.import_ecoinvent_release()`.

- It first looks for a repo-root `.env` file and loads it with `python-dotenv`.
- It expects `ECOINVENT_USERNAME` and `ECOINVENT_PASSWORD` there.
- If those values are not available, it prompts you for the credentials once.
- If the target database already exists, the cell skips the import.


In [4]:
import os
import getpass
from pathlib import Path

import bw2io as bi
from dotenv import load_dotenv


load_dotenv()

ei_version = '3.12'
system_model = 'cutoff'

username = os.environ.get('ECOINVENT_USERNAME') or input('ecoinvent username: ')
password = os.environ.get('ECOINVENT_PASSWORD') or getpass.getpass('ecoinvent password: ')

print('Username from environment:', bool(os.environ.get('ECOINVENT_USERNAME')))
print('Password from environment:', bool(os.environ.get('ECOINVENT_PASSWORD')))

Username from environment: True
Password from environment: True


In [5]:
bi.import_ecoinvent_release(
    version=ei_version,
    system_model=system_model,
    username=username,
    password=password,
)

Applying strategy: normalize_units
Applying strategy: drop_unspecified_subcategories
Applying strategy: ensure_categories_are_tuples
Applied 3 strategies in 0.01 seconds
Graph statistics for `ecoinvent-3.12-biosphere` importer:
9850 graph nodes:
	emission: 9477
	natural resource: 353
	inventory indicator: 15
	economic: 5
0 graph edges:
0 edges to the following databases:
0 unique unlinked edges (0 total):




100%|████████████████████████████████████| 9850/9850 [00:00<00:00, 37664.44it/s]

17:52:30+0200 [info     ] Vacuuming database            


Created database: ecoinvent-3.12-biosphere
Extracting XML data from 26533 datasets


/opt/homebrew/Caskroom/miniforge/base/envs/lca-course/lib/python3.11/site-packages/scikits/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__('pkg_resources').declare_namespace(__name__)
/opt/homebrew/Caskroom/miniforge/base/envs/lca-course/lib/python3.11/site-packages/scikits/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__('pkg_resources').declare_namespace(__name__)
/opt/homebrew/Caskroom/miniforge/base/envs/lca-course/lib/python3.11/site-packages/scikits/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io

17:53:16+0200 [info     ] Extracted 26533 datasets in 43.43 seconds
Applying strategy: normalize_units
Applying strategy: update_ecoinvent_locations
Applying strategy: remove_zero_amount_coproducts
Applying strategy: remove_zero_amount_inputs_with_no_activity
Applying strategy: remove_unnamed_parameters
Applying strategy: es2_assign_only_product_with_amount_as_reference_product
Applying strategy: assign_single_product_as_activity
Applying strategy: create_composite_code
Applying strategy: drop_unspecified_subcategories
Applying strategy: fix_ecoinvent_flows_pre35
Applying strategy: drop_temporary_outdated_biosphere_flows
Applying strategy: link_biosphere_by_flow_uuid
Applying strategy: link_internal_technosphere_by_composite_code
Applying strategy: delete_exchanges_missing_activity
Applying strategy: delete_ghost_exchanges
Applying strategy: remove_uncertainty_from_negative_loss_exchanges
Applying strategy: fix_unreasonably_high_lognormal_uncertainties
Applying strategy: convert_activi

100%|████████████████████████████████████| 26533/26533 [00:42<00:00, 626.05it/s]


17:54:11+0200 [info     ] Vacuuming database            
Created database: ecoinvent-3.12-cutoff


NameError: name 'source_db' is not defined

## 3) Inspect the imported database

Use a few quick checks before moving into Activity Browser.


In [6]:
source_db = 'ecoinvent-3.12-cutoff'
db = bd.Database(source_db)

print('Number of activities:', len(db))
print('Sample databases:', sorted(bd.databases)[:10])

preview = db.search('market group for electricity, medium voltage')[:5]
[(act['name'], act.get('location'), act.get('reference product')) for act in preview]


Number of activities: 26533
Sample databases: ['bafu', 'carbon fiber', 'ecoinvent-3.10-biosphere', 'ecoinvent-3.12-biosphere', 'ecoinvent-3.12-cutoff']


[('market group for electricity, medium voltage',
  'RER',
  'electricity, medium voltage'),
 ('market group for electricity, medium voltage',
  'RAF',
  'electricity, medium voltage'),
 ('market group for electricity, medium voltage',
  'BR',
  'electricity, medium voltage'),
 ('market group for electricity, medium voltage',
  'IN',
  'electricity, medium voltage'),
 ('market group for electricity, medium voltage',
  'RAS',
  'electricity, medium voltage')]

## 4) Run one compact recap LCA on an `ecoinvent` activity

We stay with the same GWP method used on Day 1 so the comparison stays easy to interpret.


In [16]:
import bw2calc as bc

method = ('ecoinvent-3.12','EF v3.1', 'climate change', 'global warming potential (GWP100)')

def pick_activity(database):
    preferred_terms = [
        ('market group for electricity, medium voltage', 'RER'),
        ('market for electricity, medium voltage', 'RER'),
        ('electricity, medium voltage', None),
    ]
    for term, preferred_location in preferred_terms:
        for activity in database.search(term):
            if preferred_location is None or activity.get('location') == preferred_location:
                return activity
    return database.random()

activity = pick_activity(db)
lca = bc.LCA({activity: 1}, method)
lca.lci()
lca.lcia()

print('Chosen activity:', activity['name'])
print('Location      :', activity.get('location'))
print('Method        :', method)
print('Score         :', lca.score)


Chosen activity: market group for electricity, medium voltage
Location      : RER
Method        : ('ecoinvent-3.12', 'EF v3.1', 'climate change', 'global warming potential (GWP100)')
Score         : 0.34950696026719025


## 5) Prepare for Activity Browser Part I

Activity Browser works best once the project contains both the BAFU exercises and the imported `ecoinvent` source database.

Before switching tools, confirm that you can point to all three of these project ingredients:

- the course project name
- the BAFU database imported on Day 1
- the `ecoinvent-3.12-cutoff` source database imported above


## Common mistakes

- Importing `ecoinvent` into a different Brightway project than `paris-lca-course-2026`.
- Using expired or incorrect `ecoinvent` credentials, or storing them in `.env` under the wrong variable names.
- Forgetting that the import can take several minutes on the first run.
- Launching Activity Browser before the import finishes writing the database.


## Recap

After this notebook, the shared Paris project should contain:

- the Day 1 BAFU material
- a working `ecoinvent-3.12-cutoff` source database
- the LCIA methods needed for later Brightway, Premise, and Activity Browser work
